# API demo

Start the server first: `MODEL_BACKEND=dummy uvicorn app.main:app --port 8000`
Set `API_BASE_URL` if the API is not local.

In [ ]:
import os, time, json
from pathlib import Path
import httpx

API = os.environ.get('API_BASE_URL', 'http://127.0.0.1:8000').rstrip('/')
SAMPLES = Path('samples')

def wait_ready(timeout=30):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            r = httpx.get(f'{API}/ready', timeout=2)
            if r.status_code == 200:
                return r.json()
        except Exception:
            pass
        time.sleep(0.4)
    raise RuntimeError('API not ready')

print(httpx.get(f'{API}/health').json())
print(wait_ready())
print(httpx.get(f'{API}/version').json())

In [ ]:
def extract(path):
    data = Path(path).read_bytes()
    r = httpx.post(f'{API}/extract', files={'file': (Path(path).name, data, 'application/pdf')}, timeout=120)
    return r

for name in ['receipt_ok.pdf', 'receipt_mismatch.pdf', 'receipt_multipage.pdf']:
    r = extract(SAMPLES / name)
    print(name, r.status_code, r.headers.get('X-Cache'), r.json().get('store_name'), r.json().get('total'))
    print('verification', r.json().get('_verification'))

In [ ]:
cases = [
    ('empty.pdf', b'', 'application/pdf'),
    ('huge.pdf', b'%PDF' + b'x' * (11 * 1024 * 1024), 'application/pdf'),
    ('photo.png', b'\x89PNG', 'image/png'),
    ('bad.pdf', b'%PDF-1.4 truncated', 'application/pdf'),
]
for name, blob, ctype in cases:
    r = httpx.post(f'{API}/extract', files={'file': (name, blob, ctype)}, timeout=30)
    print(name, r.status_code, r.json().get('error', {}).get('code'))

In [ ]:
first = extract(SAMPLES / 'receipt_ok.pdf')
second = extract(SAMPLES / 'receipt_ok.pdf')
print('first', first.headers.get('X-Cache'), 'second', second.headers.get('X-Cache'))
assert second.headers.get('X-Cache') == 'HIT'

Concurrency script (5 parallel extracts). Server must still be up.

In [ ]:
import subprocess, sys
out = subprocess.check_output([sys.executable, 'scripts/concurrency_test.py', '--base-url', API], text=True)
print(out)